<a href="https://colab.research.google.com/github/Shashank-sketchAl/ML-90-Day-Challenge-/blob/main/Day20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

df = pd.read_csv(url, sep=";")

print(df.head())
print("Dataset shape:", df.shape)
print(df.info())

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  
0      9.4        5  
1      9.8        5  
2      9.8        5 

In [3]:
print(df.columns.tolist())

['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


In [4]:
print(df.describe())

       fixed acidity  volatile acidity  citric acid  residual sugar  \
count    1599.000000       1599.000000  1599.000000     1599.000000   
mean        8.319637          0.527821     0.270976        2.538806   
std         1.741096          0.179060     0.194801        1.409928   
min         4.600000          0.120000     0.000000        0.900000   
25%         7.100000          0.390000     0.090000        1.900000   
50%         7.900000          0.520000     0.260000        2.200000   
75%         9.200000          0.640000     0.420000        2.600000   
max        15.900000          1.580000     1.000000       15.500000   

         chlorides  free sulfur dioxide  total sulfur dioxide      density  \
count  1599.000000          1599.000000           1599.000000  1599.000000   
mean      0.087467            15.874922             46.467792     0.996747   
std       0.047065            10.460157             32.895324     0.001887   
min       0.012000             1.000000         

In [5]:
print(df.isnull().sum())

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


In [6]:
print(df.duplicated().sum())
print(df["quality"].value_counts().sort_index())

240
quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64


In [7]:
df = df.drop_duplicates()

print("new dataset shape:",df.shape)
print("Remaining duplicates:", df.duplicated().sum())

new dataset shape: (1359, 12)
Remaining duplicates: 0


In [8]:
X = df.drop("quality",axis=1)
y= df["quality"]



In [9]:
from sklearn.model_selection import train_test_split
X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)



In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error,r2_score
baseline_model=Pipeline([
    ("scaler",StandardScaler()),
    ("knn",KNeighborsRegressor())
])
baseline_model.fit(X_train,y_train)

y_pred = baseline_model.predict(X_test)
print("Baseline MAE:", mean_absolute_error(y_test, y_pred))
print("Baseline R2:", r2_score(y_test, y_pred))

Baseline MAE: 0.5397058823529413
Baseline R2: 0.28251416795466255


In [12]:
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 13, 15],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

print(param_grid)

{'knn__n_neighbors': [3, 5, 7, 9, 11, 13, 15], 'knn__weights': ['uniform', 'distance'], 'knn__p': [1, 2]}


In [13]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=baseline_model,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 28 candidates, totalling 140 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('knn', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'knn__n_neighbors': [3, 5, 7, 9, 11, 13, 15],
                         'knn__p': [1, 2],
                         'knn__weights': ['uniform', 'distance']},
             scoring='neg_mean_absolute_error', verbose=1)

In [14]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV MAE:", -grid_search.best_score_)

Best Parameters: {'knn__n_neighbors': 15, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV MAE: 0.5265416056263977


In [15]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Actual values:")
print(y_test.values[:10])

print("\nPredicted values:")
print(y_pred_best[:10])

Actual values:
[5 6 7 5 4 7 5 5 6 5]

Predicted values:
[5.30618408 5.99269324 6.56828999 5.23719336 5.53617095 6.73469428
 5.19538467 5.12583589 5.85120607 5.58768939]


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred_best)
mse = mean_squared_error(y_test, y_pred_best)
r2 = r2_score(y_test, y_pred_best)

print("Optimized Model Results")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", mse ** 0.5)
print("R2 Score:", r2)

Optimized Model Results
MAE: 0.4904151854909212
MSE: 0.4122478733765065
RMSE: 0.642065318621483
R2 Score: 0.4180215111743193


In [17]:
print("Baseline MAE:", mean_absolute_error(y_test, y_pred))
print("Optimized MAE:", mae)

print("\nBaseline R2:", r2_score(y_test, y_pred))
print("Optimized R2:", r2)

Baseline MAE: 0.5397058823529413
Optimized MAE: 0.4904151854909212

Baseline R2: 0.28251416795466255
Optimized R2: 0.4180215111743193


In [18]:
results = pd.DataFrame({
    "Model": ["Baseline KNN", "Optimized KNN"],
    "MAE": [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, y_pred_best)
    ],
    "R2 Score": [
        r2_score(y_test, y_pred),
        r2_score(y_test, y_pred_best)
    ]
})

print(results)

           Model       MAE  R2 Score
0   Baseline KNN  0.539706  0.282514
1  Optimized KNN  0.490415  0.418022
